# M1M3TS - Fan Coil Unit Quick Analysis

We know some FCUs aren't working as expected.  
Our goal is 0.05 °C RMS from the target, +9ºC in this case.  
  
It would be best if you could make a command-line tool where I specify the time and we get our results—that can then be run in Notebooks. There is a DurationTime class to parse dates on the command line. 

We don't expect FCU with the heater disabled to deliver the correct temperature. 

Associated tickets:
 * [SITCOM-2122 Create script to evaluate the Fan Coil Units health](https://rubinobs.atlassian.net/browse/SITCOM-2122)

<div style="padding: 20px; background-color: #ff6600; color: white; margin-bottom: 15px;">
    The current notebook version allows querying up to 24h of data. <br>  
    Be aware that the responsiveness of the 2D Interactive Map is heavily compromised for data spanning for more than 6h. <br> 
</div>

## Input Parameters

In [ ]:
# Reference timestamp
timestamp = "2025-05-10T12:00:00"

# Time interval. Units can be "s" for seconds, "m" for minutes,
# "h" for hours, or "d" for days.
# Negative values indicate a time before the reference timestamp.
delta_t = "-6h"

# The index of the FCU (Flight Control Unit) to be used.
fcu_index = 10

# The temperature set point value for the FCU in degrees Celsius.
set_point = 10

## Setup Notebook

In [ ]:
import bqplot as bq
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import warnings

from astropy import units as u
from astropy.time import Time, TimeDelta
from astropy.time.core import TimeDeltaMissingUnitWarning
from ipywidgets import widgets
from IPython.display import HTML

from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.ts.xml.tables.m1m3.fcu_table import FCUTable

In [ ]:
# Initialize an EFD client
efd_client = makeEfdClient()

# Set global font size for labels, titles, and ticks
plt.rcParams.update(
    {
        "axes.grid": True,
        "axes.labelsize": 12,
        "axes.titlesize": 14,
        "axes.formatter.useoffset": False,
        "axes.formatter.use_mathtext": False,
        "axes.formatter.limits": (-100, 100),
        "figure.figsize": (11, 6),
        "font.size": 12,
        "grid.color": "#b0b0b0",
        "grid.linestyle": ":",
        "grid.linewidth": 0.5,
        "grid.alpha": 0.75,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
    }
)

## Helper Functions

In [ ]:
def create_url_for_summit_chronograf(t_start, t_end):
    """Create a URL for the Summit Chronograf to visualize FCU data."""
    base_url = "https://summit-lsp.lsst.codes/chronograf"
    dashboard = "sources/1/dashboards/390"

    t_start_str = t_start.isot.replace(":", "%3A")
    t_end_str = t_end.isot.replace(":", "%3A")

    url = (
        f"{base_url}/{dashboard}?refresh=Paused&lower={t_start_str}Z&upper={t_end_str}Z"
    )
    return url


def get_time_window(timestamp: str, delta_t: str):
    """Given a timestamp and a duration string, return (t_start, t_end) as
    astropy Time objects."""
    delta_t_seconds = parse_duration(delta_t)
    if delta_t_seconds <= 0:
        t_end = parse_timestamp(timestamp)
        t_start = t_end + delta_t_seconds
    else:
        t_start = parse_timestamp(timestamp)
        t_end = t_start + delta_t_seconds
    return t_start, t_end


# Function copied from lsst-ts/ts_m1m3_utils
def parse_duration(duration: str) -> TimeDelta:
    """Accept string depicting duration.

    Numbers can be suffixed with character, denomination their lengths.

    Length denominators
    -------------------
    D : days (86400 seconds)
    h : hours (3600 seconds)
    m : minutes (60 seconds)
    s : seconds (1 second)

    Examples
    --------
    '1D 1m' = 86460 seconds
    '1h 1m 30s' = 3690 seconds

    Parameters
    ----------
    duration : `str`
        Duration string. Numbers with know suffixed. Non-sufficed number will
        be treated as seconds.

    Returns
    -------
    seconds : float
        Number of seconds in string.
    """
    if not duration:
        raise ValueError("Duration string cannot be empty.")

    muls = {"D": 86400, "h": 3600, "m": 60, "s": 1, "u": 0.001, "n": 0.000001}
    ret: float = 0.0
    current: float = 0.0
    duration = duration.strip()
    sign = 1
    fraction = 0

    if duration[0] == "-":
        sign = -1
        duration = duration[1:]
    elif duration[0] == "+":
        duration = duration[1:]

    for s in duration.strip():
        if "0" <= s <= "9":
            if fraction > 0:
                current += (0.1**fraction) * int(s)
                fraction += 1
            else:
                current = current * 10 + int(s)
        elif s == ".":
            fraction = 1
        elif s == " ":
            pass
        else:
            try:
                ret += current * muls[s]
                current = 0.0
            except KeyError:
                raise ValueError(f"Unknown suffix: {s}")

    return TimeDelta(sign * (ret + current) * u.s)


def parse_timestamp(timestamp):
    """Parse the timestamp string into an astropy Time object."""
    return Time(timestamp, format="isot", scale="utc")


def plot_fcu_temperature(df, fcu_index, t_start, t_end):
    """Plot the FCU temperature data from the DataFrame."""
    title = f"FCU{fcu_index} Temperature Data\n From {t_start.iso} to {t_end.iso}"
    fig, ax = plt.subplots(num=1, clear=True)

    ax.plot(
        df[f"absoluteTemperature{fcu_index}"],
        label=f"FCU{fcu_index} Temperature",
        color="blue",
        linewidth=1.5,
    )
    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel("Temperature (deg C)")
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    ax.legend(loc="upper right", fontsize=10)

    fig.autofmt_xdate(rotation=45, ha="right")
    fig.savefig(
        f"fcu{fcu_index}_temperature_{t_start.iso.replace(':', '-')}_{t_end.iso.replace(':', '-')}.png",
        dpi=300,
    )
    plt.show()


def print_temperature_stats(df, fcu_index, set_point):
    col = f"absoluteTemperature{fcu_index}"
    stats = df[col].agg(["min", "mean", "median", "max", "std"])
    results = (
        f"FCU {fcu_index} Temperature Stats:\n"
        f"  Min:       {stats['min']:.3f} deg_C\n"
        f"  Mean:      {stats['mean']:.3f} deg_C\n"
        f"  Median:    {stats['median']:.3f} deg_C\n"
        f"  Max:       {stats['max']:.3f} deg_C\n"
        f"  Std:       {stats['std']:.3f} deg_C\n"
        f"  Set Point: {set_point:.3f} deg_C\n"
        f"  RMS:       {((df[col] - set_point) ** 2).mean() ** 0.5:.3f} deg_C"
    )
    return results


def query_fcu_data(fcu_indexes, t_start, t_end):
    """Query the EFD for FCU data within the specified time window."""
    df = getEfdData(
        efd_client,
        topic="lsst.sal.MTM1M3TS.thermalData",
        columns=["timestamp"] + [f"absoluteTemperature{i}" for i in fcu_indexes],
        begin=t_start,
        end=t_end,
    )
    return df


def fcu_quick_analysis(
    fcu_indexes,
    timestamp,
    delta_t,
    set_point,
    plot=True,
    show_url=True,
    return_html=False,
):
    """
    Perform a quick analysis of FCU temperature data.

    Parameters
    ----------
    fcu_indexes : int or list of int
        The FCU index to analyze (0-95).
        If a list is provided, the analysis will be performed for each index.
    timestamp : str
        The timestamp in ISO format (e.g., "2025-05-10T12:00:00").
    delta_t : str
        The duration string (e.g., "-10s" for 10 seconds before the timestamp).
    set_point : float
        The temperature set point for the FCU in degrees Celsius.
    plot : bool, optional
        Whether to plot the temperature data. Default is True.
    show_url : bool, optional
        Whether to show the URL for Summit Chronograf. Default is True.
    return_output: bool, optional
        Whether return the output or not. Default is False.
    """
    warnings.filterwarnings("ignore", category=TimeDeltaMissingUnitWarning)

    if isinstance(fcu_indexes, int):
        fcu_indexes = [fcu_indexes]
    elif fcu_indexes is None:
        fcu_indexes = list(range(96))
    elif not isinstance(fcu_indexes, list):
        raise ValueError("fcu_index must be an integer or a list of integers.")

    t_start, t_end = get_time_window(timestamp, delta_t)
    output = f"Time window: {t_start.iso} to {t_end.iso}"

    df = query_fcu_data(fcu_indexes, t_start, t_end)
    if df.empty:
        print("No data found for the specified time window.")
        return

    if show_url:
        url = create_url_for_summit_chronograf(t_start, t_end)
        if not return_html:
            print(f"View the data in Summit Chronograf:\n  {url}")

    for fcu_index in fcu_indexes:
        stats = print_temperature_stats(df, fcu_index, set_point)
        if plot:
            plot_fcu_temperature(df, fcu_index, t_start, t_end)

    return stats, url

## Quick Analysis

In [ ]:
html = fcu_quick_analysis(
    fcu_index, timestamp, delta_t, set_point, plot=True, show_url=True, return_html=True
)

## User Interface

In [ ]:
def extract_fcu_data(table):
    """Extract x, y positions and names from the FCU table."""
    x = [fcu.x_position for fcu in table]
    y = [fcu.y_position for fcu in table]
    names = [fcu.name for fcu in table]
    return x, y, names


def create_html_for_outputs():
    """Create an HTML widget to display the output message."""
    html = widgets.HTML(
        """
        <div id='fcu-output'>
            <h1>Select an FCU to see details.</h1>
            There is a known bug. If you click on F1, nothing will happen.<br>
            Please, click on another FCU.
        </div>'
        """
    )

    html.layout = widgets.Layout(height="270px", overflow="auto")
    return html


def create_scatter_map(_x, _y, _names):
    """Create a scatter plot for the FCU map."""
    _x_sc = bq.LinearScale()
    _y_sc = bq.LinearScale()
    _scatt = bq.Scatter(
        x=_x,
        y=_y,
        scales={"x": _x_sc, "y": _y_sc},
        default_size=150,
        enable_hover=True,
        names=_names,
        interactions={"click": "select"},
        display_names=True,
        selected_style={"fill": "red", "cursor": "pointer"},
        unselected_style={"fill": "blue", "cursor": "pointer"},
        tooltip_style={"cursor": "pointer"},
    )

    _fig = bq.Figure(
        axes=[
            bq.Axis(scale=_x_sc, visible=False),
            bq.Axis(scale=_y_sc, orientation="vertical", visible=False),
        ],
        marks=[_scatt],
        title="FCU Map",
    )

    _fig.layout.min_width = _fig.layout.min_height = "600px"
    _fig.layout.max_width = _fig.layout.max_height = "600px"

    return _scatt, _fig


def create_time_series(_df):
    """Create a time series plot for the FCU temperature data."""
    _x_sc = bq.DateScale()
    _y_sc = bq.LinearScale()
    _line = bq.Lines(
        x=_df.index,
        y=_df["absoluteTemperature0"],
        scales={"x": _x_sc, "y": _y_sc},
        labels=["absoluteTemperature0"],
    )

    _fig = bq.Figure(
        axes=[
            bq.Axis(scale=_x_sc, label="Time (HH:MM:SS)", tick_format="%H:%M:%S"),
            bq.Axis(
                scale=_y_sc,
                orientation="vertical",
                label="Temperature [dec_C]",
                label_offset="-50",
            ),
        ],
        marks=[_line],
        legend_location="top-left",
    )

    _fig.layout.min_height = _fig.layout.max_height = "330px"
    _fig.layout.min_width = _fig.layout.max_width = "600px"
    _fig.layout.display = "none"

    return _line, _fig

In [ ]:
# Unpack data from the FCUTable
x, y, names = extract_fcu_data(FCUTable)

# Replicate analysis here again
t_start, t_end = get_time_window(timestamp, delta_t)

# Get the URL for the Summit Chronograf Dashboard
url = create_url_for_summit_chronograf(t_start, t_end)

# Query the data
df = query_fcu_data(range(len(x)), t_start, t_end)

# Create HTML with the FCU statistics
details_out = create_html_for_outputs()

# Create Scatter plot
scatt, fig_scatt = create_scatter_map(x, y, names)

# Create timeline for the FCU data
line, fig_line = create_time_series(df)


# Handle click events
def on_click(change=None):
    if change is not None:
        if change["new"]:
            index = change["owner"].selected[0]

            t_start, t_end = get_time_window(timestamp, delta_t)
            stats = print_temperature_stats(df, index, set_point)

            html = f"""
            <div id='fcu-stats'>
              <h1>FCU{index + 1} Analysis (index = {index}) </h1>
              <p> 
                  <a href="{url}" style="color: blue"> Summit Chronograf - 
                  M1M3 Thermal Calibration Dashboard<br> for data between {t_start.iso} to {t_end.iso} </a>
              </p>
              <pre> {stats.replace("\n", "<br>")} </pre>
            </div>
            """

            # Test visual feedback
            details_out.value = html

            # Update plot
            fig_line.layout.display = None
            line.y = df[f"absoluteTemperature{index}"]


# Register the click event observer
scatt.observe(on_click, names=["selected"])

# Display everything
widgets.HBox([fig_scatt, widgets.VBox([details_out, fig_line])])